# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY")
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

@tool
def retrieve_game(query: str) -> str:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=5)
    docs = []
    for metadata in results["metadatas"][0]:
        docs.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
        })
    return json.dumps(docs)

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"User question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}\n\n"
        "Respond with a JSON object with two fields:\n"
        '- "useful": true if the documents contain enough information to answer the question, false otherwise\n'
        '- "description": a detailed explanation of your evaluation'
    )
    response = llm.invoke(prompt)
    try:
        report = EvaluationReport.model_validate_json(response.content)
    except Exception:
        # Fallback: parse manually if JSON extraction needed
        content = response.content or ""
        useful = "true" in content.lower() and "false" not in content.lower()
        report = EvaluationReport(useful=useful, description=content)
    return report.model_dump_json()

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> str:
    """Search the web for video game industry information.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=question, max_results=5)
    results = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in response.get("results", [])
    ]
    return json.dumps(results)

### Agent

In [ ]:
instructions = """You are UdaPlay, an expert AI research agent for the video game industry.

Your goal is to answer questions about video games accurately and helpfully.

Follow this workflow for every question:
1. Use `retrieve_game` to search the local vector database for relevant game information.
2. Use `evaluate_retrieval` to assess whether the retrieved results are sufficient to answer the question.
3. If the evaluation says the documents are NOT useful, use `game_web_search` to find the answer online.
4. Provide a clear, concise, and accurate final answer to the user.

Always base your answer on the best available information from the tools.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [ ]:
def print_agent_trace(query: str):
    """Run the agent and print full reasoning trace: tool calls, args, results, and final answer."""
    print(f"\n{'='*60}")
    print(f"Q: {query}")
    print(f"{'='*60}")

    run = agent.invoke(query)
    messages = run.get_final_state()["messages"]

    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"\n  [Tool Call] {tc.function.name}")
                for k, v in args.items():
                    # Truncate long values for readability
                    display = str(v)[:120] + "..." if len(str(v)) > 120 else str(v)
                    print(f"    {k}: {display}")

        elif isinstance(msg, ToolMessage):
            result = msg.content
            # If it's JSON, try to pretty-print a summary
            try:
                parsed = json.loads(result)
                if isinstance(parsed, list):
                    print(f"  [Tool Result] {msg.name} → {len(parsed)} item(s) returned")
                    for item in parsed[:2]:  # show first 2 items
                        if isinstance(item, dict):
                            # Print URL if it's a web result, else game name
                            display = item.get("url") or item.get("Name") or str(item)[:80]
                            print(f"    • {display}")
                    if len(parsed) > 2:
                        print(f"    • ... ({len(parsed) - 2} more)")
                elif isinstance(parsed, dict):
                    print(f"  [Tool Result] {msg.name} → useful={parsed.get('useful')} | {str(parsed.get('description',''))[:100]}...")
            except Exception:
                print(f"  [Tool Result] {msg.name} → {result[:150]}")

        elif isinstance(msg, AIMessage) and msg.content:
            # Skip the system/user messages, only show final AI answer
            pass

    # Final answer
    answer = next(
        (m.content for m in reversed(messages) if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )
    print(f"\n  [Answer] {answer}")


queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for query in queries:
    print_agent_trace(query)

### (Optional) Advanced

In [ ]:
## Option 1: Long-Term Memory
#
# LongTermMemory uses VectorStoreManager (in-memory ChromaDB + OpenAI embeddings)
# to store and semantically recall game facts across agent turns.
#
# Two new tools are added:
#   - save_game_fact  : stores a fact learned during a conversation
#   - recall_game_facts : retrieves previously saved facts relevant to the current query

from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

MEMORY_OWNER = "udaplay_user"

_ltm_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
long_term_memory = LongTermMemory(db=_ltm_manager)

@tool
def save_game_fact(fact: str) -> str:
    """Save an interesting or useful game fact to long-term memory for future reference.
    args:
    - fact: a concise fact about a video game worth remembering.
    """
    fragment = MemoryFragment(content=fact, owner=MEMORY_OWNER, namespace="game_facts")
    long_term_memory.register(fragment)
    return f"Saved: {fact}"

@tool
def recall_game_facts(query: str) -> str:
    """Recall previously saved game facts that are relevant to the current query.
    args:
    - query: a question or topic to search for in stored game facts.
    """
    result = long_term_memory.search(
        query_text=query,
        owner=MEMORY_OWNER,
        namespace="game_facts",
        limit=3,
    )
    if not result.fragments:
        return "No relevant facts found in long-term memory."
    return json.dumps([f.content for f in result.fragments])


memory_instructions = """You are UdaPlay, an expert AI research agent for the video game industry.

Follow this workflow for every question:
1. Use `recall_game_facts` to check long-term memory for a previously learned answer.
2. Use `retrieve_game` to search the local vector database.
3. Use `evaluate_retrieval` to assess whether retrieved results are sufficient.
4. If not useful, use `game_web_search` to search the web.
5. Give a clear, accurate answer.
6. Use `save_game_fact` to store any new interesting fact you discovered.
"""

memory_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=memory_instructions,
    tools=[recall_game_facts, retrieve_game, evaluate_retrieval, game_web_search, save_game_fact],
    temperature=0.0,
)

# First query — nothing in memory yet, will retrieve + save
run1 = memory_agent.invoke("When was Pokémon Gold and Silver released?")
state1 = run1.get_final_state()
print("Turn 1:", next((m.content for m in reversed(state1["messages"]) if isinstance(m, AIMessage) and m.content), ""))

# Second query — recall_game_facts should surface the saved Pokémon fact
run2 = memory_agent.invoke("Tell me something about Pokémon Gold and Silver.")
state2 = run2.get_final_state()
print("Turn 2:", next((m.content for m in reversed(state2["messages"]) if isinstance(m, AIMessage) and m.content), ""))

In [ ]:
## Option 2: State Machine with Tool Nodes
#
# Instead of the Agent's generic "call LLM → maybe run tools" loop,
# we hard-wire the research pipeline as named StateMachine nodes:
#
#   entry → retrieve_node → evaluate_node → [answer_node | web_search_node] → answer_node → termination
#
# Each node calls exactly one operation, making the flow explicit and auditable.

from typing import TypedDict, Optional as Opt
from lib.state_machine import StateMachine, Step, EntryPoint, Termination

class UdaPlayState(TypedDict):
    query: str
    retrieved_docs: Opt[str]
    evaluation: Opt[str]
    web_results: Opt[str]
    answer: Opt[str]


def _retrieve_node(state: UdaPlayState) -> dict:
    docs = retrieve_game(state["query"])
    return {"retrieved_docs": docs}

def _evaluate_node(state: UdaPlayState) -> dict:
    evaluation = evaluate_retrieval(state["query"], state["retrieved_docs"])
    return {"evaluation": evaluation}

def _web_search_node(state: UdaPlayState) -> dict:
    results = game_web_search(state["query"])
    return {"web_results": results}

def _answer_node(state: UdaPlayState) -> dict:
    llm = LLM(model="gpt-4o-mini", temperature=0.0)
    context = state.get("web_results") or state.get("retrieved_docs") or ""
    prompt = (
        f"Answer the following question using the provided context.\n\n"
        f"Question: {state['query']}\n\n"
        f"Context:\n{context}\n\n"
        "Give a concise, accurate answer."
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}


def _route_after_evaluate(state: UdaPlayState):
    """Go to web_search if the local docs were not useful, else skip to answer."""
    try:
        report = EvaluationReport.model_validate_json(state["evaluation"])
        if not report.useful:
            return web_search_node
    except Exception:
        pass
    return answer_node


# Build the state machine
sm = StateMachine[UdaPlayState](UdaPlayState)

entry          = EntryPoint[UdaPlayState]()
retrieve_node  = Step[UdaPlayState]("retrieve",   _retrieve_node)
evaluate_node  = Step[UdaPlayState]("evaluate",   _evaluate_node)
web_search_node = Step[UdaPlayState]("web_search", _web_search_node)
answer_node    = Step[UdaPlayState]("answer",     _answer_node)
termination    = Termination[UdaPlayState]()

sm.add_steps([entry, retrieve_node, evaluate_node, web_search_node, answer_node, termination])

sm.connect(entry,          retrieve_node)
sm.connect(retrieve_node,  evaluate_node)
sm.connect(evaluate_node,  [web_search_node, answer_node], _route_after_evaluate)
sm.connect(web_search_node, answer_node)
sm.connect(answer_node,    termination)


def run_state_machine_agent(query: str) -> str:
    initial: UdaPlayState = {
        "query": query,
        "retrieved_docs": None,
        "evaluation": None,
        "web_results": None,
        "answer": None,
    }
    run = sm.run(initial)
    return run.get_final_state()["answer"]


print("=== State Machine Agent ===")
for q in [
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]:
    print(f"\nQ: {q}")
    print(f"A: {run_state_machine_agent(q)}")

### Excellence: JSON Structured Output + Retrieval Visualization

In [ ]:
## JSON Structured Output
#
# The agent now returns two things for every query:
#   - natural_language: a human-readable answer (as before)
#   - structured: a GameAnswer object with typed fields for downstream use
#
# We use OpenAI's structured output mode (LLM.invoke(response_format=GameAnswer))
# so the JSON is guaranteed to match the schema — no parsing failures.

from typing import List as TList
from pydantic import BaseModel, Field

class GameAnswer(BaseModel):
    natural_language: str = Field(description="Clear, human-readable answer to the question")
    games_mentioned: TList[str] = Field(description="Names of games referenced in the answer")
    source: str = Field(description="Where the answer came from: 'rag', 'web', or 'memory'")
    year_released: str = Field(description="Release year(s) if relevant, else empty string")
    confidence: str = Field(description="high / medium / low — how confident the answer is")
    citation: str = Field(description="Game name + platform + year, or URL if from web search")


def structured_agent_answer(query: str, context: str, source: str) -> GameAnswer:
    """Ask the LLM to produce a GameAnswer given the retrieved context."""
    llm = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = (
        f"Answer the following video game question using the provided context.\n\n"
        f"Question: {query}\n\n"
        f"Context (source={source}):\n{context}\n\n"
        "Fill in every field of the schema accurately."
    )
    response = llm.invoke(prompt, response_format=GameAnswer)
    # response_format with a Pydantic model returns the parsed object in .parsed
    if hasattr(response, "parsed") and response.parsed:
        return response.parsed
    # fallback: parse from content string
    return GameAnswer.model_validate_json(response.content)


def run_with_structured_output(query: str):
    """Full pipeline: retrieve → evaluate → (web if needed) → structured + NL answer."""
    # Step 1: retrieve
    raw_docs = retrieve_game(query)

    # Step 2: evaluate
    eval_json = evaluate_retrieval(query, raw_docs)
    eval_report = EvaluationReport.model_validate_json(eval_json)

    # Step 3: choose source
    if eval_report.useful:
        context, source = raw_docs, "rag"
    else:
        context, source = game_web_search(query), "web"

    # Step 4: structured answer
    answer: GameAnswer = structured_agent_answer(query, context, source)

    print(f"\nQ: {query}")
    print(f"  Natural language : {answer.natural_language}")
    print(f"  Structured JSON  : {answer.model_dump_json(indent=2)}")
    return answer


for q in [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]:
    run_with_structured_output(q)

In [ ]:
## Retrieval Process Visualization
#
# For a given query this shows:
#   1. Top-N retrieved documents ranked by cosine similarity (bar chart)
#   2. The evaluation verdict (RAG used vs. web fallback triggered)
#   3. The pipeline path taken (colour-coded steps)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def visualize_retrieval(query: str, n_results: int = 5):
    # ── 1. Retrieve with distances ────────────────────────────────────────────
    raw = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )
    metadatas = raw["metadatas"][0]
    distances = raw["distances"][0]

    # ChromaDB returns L2 distances; convert to cosine similarity (0–1 scale)
    similarities = [max(0.0, 1 - (d / 2)) for d in distances]
    labels = [f"{m['Name']}\n({m['Platform']}, {m['YearOfRelease']})" for m in metadatas]

    # ── 2. Evaluate retrieval ─────────────────────────────────────────────────
    docs_json = json.dumps([
        {"Name": m["Name"], "Platform": m["Platform"],
         "YearOfRelease": m["YearOfRelease"], "Description": m["Description"]}
        for m in metadatas
    ])
    eval_json = evaluate_retrieval(query, docs_json)
    eval_report = EvaluationReport.model_validate_json(eval_json)
    rag_used = eval_report.useful

    # ── 3. Plot ───────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={"width_ratios": [3, 1]})
    fig.suptitle(f'Retrieval Process\n"{query}"', fontsize=12, fontweight="bold")

    # — Left: similarity bar chart —
    ax = axes[0]
    y_pos = np.arange(len(labels))
    colors = ["#2ecc71" if s >= 0.75 else "#f39c12" if s >= 0.55 else "#e74c3c"
              for s in similarities]
    bars = ax.barh(y_pos, similarities, color=colors, edgecolor="white", height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Cosine Similarity")
    ax.set_xlim(0, 1.05)
    ax.axvline(0.75, color="#2ecc71", linestyle="--", linewidth=1, alpha=0.6, label="High (≥0.75)")
    ax.axvline(0.55, color="#f39c12", linestyle="--", linewidth=1, alpha=0.6, label="Medium (≥0.55)")
    ax.invert_yaxis()
    ax.set_title("Retrieved Documents by Similarity", fontsize=10)
    for bar, sim in zip(bars, similarities):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{sim:.2f}", va="center", fontsize=8)
    green_p  = mpatches.Patch(color="#2ecc71", label="High (≥0.75)")
    orange_p = mpatches.Patch(color="#f39c12", label="Medium (≥0.55)")
    red_p    = mpatches.Patch(color="#e74c3c", label="Low (<0.55)")
    ax.legend(handles=[green_p, orange_p, red_p], fontsize=8, loc="lower right")

    # — Right: pipeline path —
    ax2 = axes[1]
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis("off")
    ax2.set_title("Pipeline Path", fontsize=10)

    steps = [
        ("Query",      0.88, "#3498db"),
        ("Retrieve",   0.72, "#3498db"),
        ("Evaluate",   0.56, "#3498db"),
        ("RAG Answer"  if rag_used else "Web Search", 0.40,
         "#2ecc71"     if rag_used else "#e74c3c"),
        ("Answer",     0.24, "#2ecc71"),
    ]
    for label, y, color in steps:
        ax2.add_patch(mpatches.FancyBboxPatch((0.15, y - 0.06), 0.7, 0.1,
                      boxstyle="round,pad=0.02", facecolor=color,
                      edgecolor="white", alpha=0.85))
        ax2.text(0.5, y - 0.01, label, ha="center", va="center",
                 color="white", fontsize=9, fontweight="bold")
        if y > 0.24:
            ax2.annotate("", xy=(0.5, y - 0.065), xytext=(0.5, y - 0.12),
                         arrowprops=dict(arrowstyle="->", color="#555", lw=1.5))

    verdict = "✓ RAG sufficient" if rag_used else "✗ Fallback to web"
    verdict_color = "#2ecc71" if rag_used else "#e74c3c"
    ax2.text(0.5, 0.08, verdict, ha="center", va="center",
             color=verdict_color, fontsize=9, fontstyle="italic")

    plt.tight_layout()
    plt.show()
    print(f"  Evaluation: {eval_report.description[:120]}...")


# Visualize for all three sample queries
for q in [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]:
    visualize_retrieval(q)